In [ ]:
# #Parquet Files 
#         a) Column oriented - Each column data stored as seperate files at backend
#         b) Data Encrypted - Cannot see data like csv and txt

# When you write query like 'select * from department where deptId=30' then it will scan deptid column for 30 and retrieve those rows data only
# while row oriented scans each and every row data and retrieve matched records

# Column oriented - Vertical partition
#                 - supports OLAP and doesnot support OLTP
#                 - Physical storage like
#                   id,1,2,3,name,happy,sad,joy
#                 - Row insertion be like each and every table/file has tobe updated with new data 
#                 - Performance-wise, its good for read operations

# Row Oriented    - Horizontal Partition
#                 - Supports OLTP does not support OLAP
#                 - Physical storage like 
#                   id Name
#                   1  Happy
#                   2  Sad
#                   3  Joy
#                  - Row insertion be like append rows data at the end of the file
#                  - Performance-wise, its good for write operations

# We cannot directly create parquet fie just have convert dataframe to parquet files


In [1]:
#Read employees data

df=spark.read.csv(
    'abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Files/empdetails.csv',
    header=True
)

display(df)

StatementMeta(, 2d4a2e4c-707d-4249-9795-b0df483c4607, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9be637a4-d816-4356-9d50-ef6de2246301)

In [2]:
df1=df.select('id','name','salary','gender')
path='abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Files/emp'
df1.write.format('parquet').mode('append').save(path)

StatementMeta(, 2d4a2e4c-707d-4249-9795-b0df483c4607, 4, Finished, Available, Finished, False)

In [3]:
#Part file created with .snappy.parquet extension - Here snappy is compression type like zip
notebookutils.fs.ls(path)

StatementMeta(, 2d4a2e4c-707d-4249-9795-b0df483c4607, 5, Finished, Available, Finished, False)

[FileInfo(path=abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Files/emp/_SUCCESS, name=_SUCCESS, size=0),
 FileInfo(path=abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Files/emp/part-00000-c7063ea1-a8e3-40e0-9ff3-54bec7afdfcc-c000.snappy.parquet, name=part-00000-c7063ea1-a8e3-40e0-9ff3-54bec7afdfcc-c000.snappy.parquet, size=2069)]

In [5]:
path='abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Files/emp/'

df=spark.read.parquet(path)
display(df)

StatementMeta(, 2d4a2e4c-707d-4249-9795-b0df483c4607, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 12964a58-fe61-44b0-b91f-6fdf32343120)

In [8]:
df.write.format('delta').mode('append').save(path)

StatementMeta(, 2d4a2e4c-707d-4249-9795-b0df483c4607, 12, Finished, Available, Finished, False)

In [9]:
notebookutils.fs.ls(path)

StatementMeta(, 2d4a2e4c-707d-4249-9795-b0df483c4607, 13, Finished, Available, Finished, False)

[FileInfo(path=abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Files/emp/_SUCCESS, name=_SUCCESS, size=0),
 FileInfo(path=abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Files/emp/_delta_log, name=_delta_log, size=0),
 FileInfo(path=abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Files/emp/part-00000-c7063ea1-a8e3-40e0-9ff3-54bec7afdfcc-c000.snappy.parquet, name=part-00000-c7063ea1-a8e3-40e0-9ff3-54bec7afdfcc-c000.snappy.parquet, size=2069),
 FileInfo(path=abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Files/emp/part-00000-dcf2a2db-2674-456a-94b3-be5d77148cac-c000.snappy.parquet, name=part-00000-dcf2a2db-2674-456a-94b3-be5d77148cac-c000.snappy.parquet, size=2368)]

In [12]:
#Delta and Parquet both are same - Delta format file maintain delta log while parquet not
# DML command does not work on parquet file while in delta works
# as delta does not directly change the record and it maintains log and creates new record for it
# also, it scans log fiest for any read operation and give result

df=spark.read.parquet(path)
display(df)

StatementMeta(, 2d4a2e4c-707d-4249-9795-b0df483c4607, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fd7ac1dd-e129-40fa-ac5e-5cc6f47a7679)

In [13]:
df.write.format('parquet').mode('append').saveAsTable('emp_dtls_parquet')
df.write.format('delta').mode('append').saveAsTable('emp_dtls_parquetdelta')

StatementMeta(, 2d4a2e4c-707d-4249-9795-b0df483c4607, 17, Finished, Available, Finished, False)

In [15]:
df=spark.sql('select * from emp_dtls_parquet')
df1=spark.sql('select * from emp_dtls_parquetdelta')
display(df)
display(df1)

StatementMeta(, 2d4a2e4c-707d-4249-9795-b0df483c4607, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c224309c-737c-4718-99f7-ee90cd603677)

SynapseWidget(Synapse.DataFrame, 874a4af5-1e0d-418f-b679-20da20dcec0f)

In [17]:
spark.sql("""insert into emp_dtls_parquet values('109','hymaa','400000','F')""")
spark.sql("""insert into emp_dtls_parquetdelta values('109','hymaa','400000','F')""")

df=spark.sql('select * from emp_dtls_parquet')
df1=spark.sql('select * from emp_dtls_parquetdelta')
display(df)
display(df1)

StatementMeta(, 2d4a2e4c-707d-4249-9795-b0df483c4607, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fc46e65e-b557-43be-b725-d720d3e9ac51)

SynapseWidget(Synapse.DataFrame, 1399c4d2-cc1b-4d2c-ba18-6fe79390af40)

In [20]:
display(spark.sql('desc formatted emp_dtls_parquet'))
display(spark.sql('desc formatted emp_dtls_parquetdelta'))

StatementMeta(, 2d4a2e4c-707d-4249-9795-b0df483c4607, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 965e3f63-1467-4062-aa40-cb2cc11cceec)

SynapseWidget(Synapse.DataFrame, 1ec25456-8590-4078-a4b5-8a0e4a5a5b5d)

In [24]:
spark.sql("""UPDATE emp_dtls_parquet set id='1091' where id='109'""")
df=spark.sql('select * from emp_dtls_parquet')
display(df)


StatementMeta(, 2d4a2e4c-707d-4249-9795-b0df483c4607, 28, Finished, Available, Finished, False)

UnsupportedOperationException: UPDATE TABLE is not supported temporarily.

In [25]:
spark.sql("""UPDATE emp_dtls_parquetdelta set id='1091' where id='109'""")
df1=spark.sql('select * from emp_dtls_parquetdelta')
display(df1)

StatementMeta(, 2d4a2e4c-707d-4249-9795-b0df483c4607, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6669e3c2-f133-4db4-8caf-1cebdb503baf)